# SOC-Aid MVP\n\n> Evidence-aware Agentic AI for Security Alert Triage\n\nThis clean notebook presents the final SOC-Aid workflow: alert parsing, correlation, authentication evidence, explainable risk assessment, evidence-aware LLM analysis, analyst recommendations, human oversight, verification, and a Gradio demo.\n

## 1. Environment Setup

In [ ]:
# 1. SOC-Aid | Environment Setup

!pip install -q langchain langchain-groq langgraph pandas

import os
import json
import pandas as pd

from typing import TypedDict, List, Dict, Any
from google.colab import userdata

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END


# Secure API key from Google Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found. Add it to Google Colab → Secrets."
    )


# LLM Configuration
MODEL_NAME = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    api_key=GROQ_API_KEY
)

print(f"✓ Groq LLM connected: {MODEL_NAME}")
print("✓ SOC-Aid environment initialized successfully")

✓ Groq LLM connected: openai/gpt-oss-20b
✓ SOC-Aid environment initialized successfully


## 2. Security Alert Data and Core Tools

In [ ]:
# 2. SOC-Aid | Security Alert Data & Tools

# Sample Security Alerts

alerts = [
    {
        "alert_id": "A001",
        "timestamp": "2026-08-18 01:42:10",
        "user": "admin",
        "source_ip": "185.220.101.45",
        "event_type": "failed_login",
        "message": "Multiple failed login attempts detected",
        "severity": "medium"
    },
    {
        "alert_id": "A002",
        "timestamp": "2026-08-18 01:45:32",
        "user": "admin",
        "source_ip": "185.220.101.45",
        "event_type": "successful_login",
        "message": "Successful login after multiple failed attempts",
        "severity": "high"
    },
    {
        "alert_id": "A003",
        "timestamp": "2026-08-18 02:10:05",
        "user": "employee01",
        "source_ip": "192.168.1.25",
        "event_type": "normal_login",
        "message": "Normal employee login",
        "severity": "low"
    },
    {
        "alert_id": "A004",
        "timestamp": "2026-08-18 02:15:44",
        "user": "employee02",
        "source_ip": "10.0.0.15",
        "event_type": "malware_detected",
        "message": "Malicious file detected on endpoint",
        "severity": "critical"
    }
]

alerts_df = pd.DataFrame(alerts)


# Tool 1: Alert Parser

def parse_alert(alert: Dict[str, Any]) -> Dict[str, Any]:
    """
    Validate and normalize a security alert.
    """

    required_fields = [
        "alert_id",
        "timestamp",
        "user",
        "source_ip",
        "event_type",
        "message",
        "severity"
    ]

    missing_fields = [
        field for field in required_fields
        if field not in alert
        or alert[field] is None
        or str(alert[field]).strip() == ""
    ]

    if missing_fields:
        raise ValueError(
            f"Alert is missing required fields: {missing_fields}"
        )

    valid_severities = {
        "low",
        "medium",
        "high",
        "critical"
    }

    severity = str(alert["severity"]).lower().strip()

    if severity not in valid_severities:
        raise ValueError(
            f"Invalid severity '{severity}'. "
            f"Expected one of: {sorted(valid_severities)}"
        )

    return {
        "alert_id": str(alert["alert_id"]).strip(),
        "timestamp": str(alert["timestamp"]).strip(),
        "user": str(alert["user"]).strip(),
        "source_ip": str(alert["source_ip"]).strip(),
        "event_type": str(alert["event_type"]).strip(),
        "message": str(alert["message"]).strip(),
        "severity": severity
    }


# Tool 2: Alert Correlation

def find_related_alerts(
    alert: Dict[str, Any],
    history: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Find alerts related to the current alert
    using user or source IP correlation.
    """

    related = []

    for previous_alert in history:

        if previous_alert["alert_id"] == alert["alert_id"]:
            continue

        same_user = (
            previous_alert["user"] == alert["user"]
        )

        same_ip = (
            previous_alert["source_ip"] == alert["source_ip"]
        )

        if same_user or same_ip:
            related.append(previous_alert)

    return related


# Improved Risk Assessment

def assess_risk(
    alert: Dict[str, Any],
    related_alerts: List[Dict[str, Any]]
) -> Dict[str, Any]:

    score = 0
    reasons = []
    risk_factors = []

    severity_scores = {
        "low": 10,
        "medium": 30,
        "high": 60,
        "critical": 90
    }

    # Severity

    severity_points = severity_scores.get(
        alert["severity"],
        0
    )

    score += severity_points

    risk_factors.append({
        "factor": f"Alert severity: {alert['severity']}",
        "points": severity_points
    })

    # Related Activity

    if related_alerts:

        score += 10

        reasons.append(
            "Related security activity was found."
        )

        risk_factors.append({
            "factor": "Related security activity",
            "points": 10
        })

    # Failed Login → Successful Login Pattern

    failed_logins = sum(
        1
        for item in related_alerts
        if item["event_type"] == "failed_login"
    )

    if (
        failed_logins >= 1
        and alert["event_type"] == "successful_login"
    ):

        score += 20

        reasons.append(
            "A successful login occurred after "
            "failed login attempts."
        )

        risk_factors.append({
            "factor": "Successful login after failed login attempts",
            "points": 20
        })

    # Cap Score

    score = min(score, 100)

    # Risk Level

    if score >= 80:
        risk_level = "CRITICAL"

    elif score >= 60:
        risk_level = "HIGH"

    elif score >= 30:
        risk_level = "MEDIUM"

    else:
        risk_level = "LOW"

    return {
        "risk_score": score,
        "risk_level": risk_level,
        "reasons": reasons,
        "risk_factors": risk_factors
    }


# Tool Status

print("✓ Security alerts loaded")
print(f"✓ Total alerts available: {len(alerts_df)}")
print("✓ Alert parsing tool ready")
print("✓ Alert correlation tool ready")
print("✓ Explainable risk assessment tool ready")

✓ Security alerts loaded
✓ Total alerts available: 4
✓ Alert parsing tool ready
✓ Alert correlation tool ready
✓ Explainable risk assessment tool ready


## 3. Authentication Evidence Test Data

In [ ]:
# ============================================================
# SOC-Aid | Add Authentication Attack Test Data
# ============================================================

from datetime import datetime, timedelta

# Keep the existing alerts
original_alert_count = len(alerts)

# Target user and IP
test_user = "employee01"
test_ip = "192.168.1.25"

# Starting time for the failed-login sequence
start_time = datetime.strptime(
    "2026-08-19 10:20:00",
    "%Y-%m-%d %H:%M:%S"
)

# Add 50 failed login attempts within 10 minutes
for i in range(50):

    failed_alert = {
        "alert_id": f"AUTH-FAIL-{i+1:03d}",
        "timestamp": (
            start_time + timedelta(seconds=i * 10)
        ).strftime("%Y-%m-%d %H:%M:%S"),
        "user": test_user,
        "source_ip": test_ip,
        "event_type": "failed_login",
        "message": "Failed authentication attempt",
        "severity": "medium"
    }

    alerts.append(failed_alert)

print("=" * 70)
print("       SOC-Aid AUTHENTICATION TEST DATA ADDED")
print("=" * 70)

print(f"\nOriginal alerts : {original_alert_count}")
print(f"Added failed logins : 50")
print(f"Total alerts now : {len(alerts)}")

print("\nTest user       :", test_user)
print("Test source IP  :", test_ip)
print("Time window     : 10 minutes")
print("Failed attempts : 50")

print("\n✓ Existing alerts preserved")
print("✓ 50 failed-login events added")
print("✓ Ready for correlation testing")

print("=" * 70)

       SOC-Aid AUTHENTICATION TEST DATA ADDED

Original alerts : 4
Added failed logins : 50
Total alerts now : 54

Test user       : employee01
Test source IP  : 192.168.1.25
Time window     : 10 minutes
Failed attempts : 50

✓ Existing alerts preserved
✓ 50 failed-login events added
✓ Ready for correlation testing


## 4. Workflow State and Authentication Correlation

In [ ]:
# ============================================================
# SOC-Aid | Workflow State
# ============================================================

from typing import Dict, Any, List, TypedDict


class SOCAidState(TypedDict, total=False):

    # Original alert
    alert: Dict[str, Any]

    # Parsed alert
    parsed_alert: Dict[str, Any]

    # Correlated alerts
    related_alerts: List[Dict[str, Any]]

    # NEW: Authentication evidence
    authentication_evidence: Dict[str, Any]

    # Risk assessment
    risk_assessment: Dict[str, Any]

    # AI analysis
    analysis: str

    # Recommendation
    recommendation: str

    # Final human decision
    decision: str

    # Error handling
    error: str


print("=" * 70)
print("       SOC-Aid STATE UPDATED")
print("=" * 70)

print("✓ authentication_evidence added to SOCAidState")
print("✓ risk_assessment preserved")
print("✓ AI analysis preserved")
print("✓ recommendation preserved")
print("✓ human decision preserved")
print("=" * 70)

       SOC-Aid STATE UPDATED
✓ authentication_evidence added to SOCAidState
✓ risk_assessment preserved
✓ AI analysis preserved
✓ recommendation preserved
✓ human decision preserved


In [ ]:
# ============================================================
# SOC-Aid | 10-Minute Authentication Correlation
# ============================================================

from datetime import datetime

def find_authentication_pattern(
    alert,
    history,
    window_minutes=10
):
    alert_time = datetime.strptime(
        alert["timestamp"],
        "%Y-%m-%d %H:%M:%S"
    )

    related_failures = []

    for item in history:

        # Ignore the current alert
        if item["alert_id"] == alert["alert_id"]:
            continue

        # Only failed-login events
        if item.get("event_type") != "failed_login":
            continue

        # Same user AND same source IP
        if item.get("user") != alert.get("user"):
            continue

        if item.get("source_ip") != alert.get("source_ip"):
            continue

        item_time = datetime.strptime(
            item["timestamp"],
            "%Y-%m-%d %H:%M:%S"
        )

        # Difference between failed login and current alert
        difference = alert_time - item_time

        # Failure must occur BEFORE the successful login
        if difference.total_seconds() < 0:
            continue

        # Failure must be within configured time window
        if difference.total_seconds() <= window_minutes * 60:
            related_failures.append(item)

    return related_failures




In [ ]:
# ============================================================
# SOC-Aid | AUTHENTICATION EVIDENCE BUILDER
# ============================================================

def build_authentication_evidence(alert, alerts, window_minutes=10):

    failed_logins = find_authentication_pattern(
        alert,
        alerts,
        window_minutes=window_minutes
    )

    evidence = {
        "failed_login_count": len(failed_logins),
        "time_window_minutes": window_minutes,
        "pattern_detected": (
            len(failed_logins) > 0
            and alert.get("event_type") == "successful_login"
        ),
        "first_failed_login": (
            failed_logins[0]["timestamp"]
            if failed_logins else None
        ),
        "last_failed_login": (
            failed_logins[-1]["timestamp"]
            if failed_logins else None
        )
    }

    return evidence


## 5. Final Agent Architecture

In [ ]:
# ============================================================
# SOC-Aid | FINAL INTEGRATED LANGGRAPH AGENT
# Authentication Evidence + Risk + AI + Human Oversight
# ============================================================

def build_soc_aid_agent(
    llm,
    alerts: List[Dict[str, Any]]
):
    """
    Build the final SOC-Aid LangGraph workflow.

    Workflow:
        Parse
          ↓
        Correlate
          ↓
        Authentication Evidence
          ↓
        Risk Assessment
          ↓
        Evidence-Aware AI Analysis
          ↓
        Recommendation
          ↓
        Human Analyst Decision
    """

    # ========================================================
    # NODE 1 — PARSE ALERT
    # ========================================================

    def parse_alert_node(
        state: SOCAidState
    ) -> SOCAidState:

        try:
            parsed = parse_alert(state["alert"])

            return {
                **state,
                "parsed_alert": parsed,
                "error": ""
            }

        except Exception as e:

            return {
                **state,
                "error": f"Alert parsing failed: {str(e)}"
            }

    # ========================================================
    # NODE 2 — CORRELATE ALERTS
    # ========================================================

    def correlation_node(
        state: SOCAidState
    ) -> SOCAidState:

        if state.get("error"):
            return state

        try:

            related = find_related_alerts(
                state["parsed_alert"],
                alerts
            )

            return {
                **state,
                "related_alerts": related
            }

        except Exception as e:

            return {
                **state,
                "error":
                    f"Alert correlation failed: {str(e)}"
            }

    # ========================================================
    # NODE 3 — AUTHENTICATION EVIDENCE
    # ========================================================

    def authentication_evidence_node(
        state: SOCAidState
    ) -> SOCAidState:

        if state.get("error"):
            return state

        try:

            evidence = build_authentication_evidence(
                state["parsed_alert"],
                alerts,
                window_minutes=10
            )

            return {
                **state,
                "authentication_evidence": evidence
            }

        except Exception as e:

            return {
                **state,
                "error":
                    "Authentication evidence generation "
                    f"failed: {str(e)}"
            }

    # ========================================================
    # NODE 4 — RISK ASSESSMENT
    # ========================================================

    def risk_node(
        state: SOCAidState
    ) -> SOCAidState:

        if state.get("error"):
            return state

        try:

            alert = state["parsed_alert"]
            related = state["related_alerts"]
            evidence = state["authentication_evidence"]

            score = 0
            reasons = []
            risk_factors = []

            # ------------------------------------------------
            # Alert severity
            # ------------------------------------------------

            severity_scores = {
                "low": 10,
                "medium": 30,
                "high": 60,
                "critical": 90
            }

            severity_points = severity_scores.get(
                alert.get("severity", "low"),
                0
            )

            score += severity_points

            risk_factors.append({
                "factor":
                    f"Alert severity: "
                    f"{alert.get('severity')}",
                "points": severity_points
            })

            # ------------------------------------------------
            # Related activity
            # ------------------------------------------------

            if related:

                score += 10

                reasons.append(
                    "Related security activity was found."
                )

                risk_factors.append({
                    "factor":
                        "Related security activity",
                    "points": 10
                })

            # ------------------------------------------------
            # Authentication evidence
            # ------------------------------------------------

            failed_count = evidence.get(
                "failed_login_count",
                0
            )

            pattern_detected = evidence.get(
                "pattern_detected",
                False
            )

            # 50+ failed attempts
            if failed_count >= 50:

                score += 30

                reasons.append(
                    f"{failed_count} failed login attempts "
                    "were detected within the configured "
                    "time window."
                )

                risk_factors.append({
                    "factor":
                        "50+ failed login attempts",
                    "points": 30
                })

            # Successful login after failures
            if (
                failed_count >= 1
                and alert.get("event_type")
                    == "successful_login"
            ):

                score += 20

                reasons.append(
                    "A successful login occurred after "
                    "failed login attempts."
                )

                risk_factors.append({
                    "factor":
                        "Successful login after "
                        "failed login attempts",
                    "points": 20
                })

            # ------------------------------------------------
            # Cap score
            # ------------------------------------------------

            score = min(score, 100)

            # ------------------------------------------------
            # Risk level
            # ------------------------------------------------

            if score >= 80:
                risk_level = "CRITICAL"

            elif score >= 60:
                risk_level = "HIGH"

            elif score >= 30:
                risk_level = "MEDIUM"

            else:
                risk_level = "LOW"

            risk = {
                "risk_score": score,
                "risk_level": risk_level,
                "reasons": reasons,
                "risk_factors": risk_factors
            }

            return {
                **state,
                "risk_assessment": risk
            }

        except Exception as e:

            return {
                **state,
                "error":
                    f"Risk assessment failed: {str(e)}"
            }

    # ========================================================
    # NODE 5 — EVIDENCE-AWARE AI ANALYSIS
    # ========================================================

    def analysis_node(
        state: SOCAidState
    ) -> SOCAidState:

        if state.get("error"):
            return state

        try:

            alert = state["parsed_alert"]
            related = state["related_alerts"]
            risk = state["risk_assessment"]
            evidence = state["authentication_evidence"]

            prompt = f"""
You are SOC-Aid, an explainable AI assistant
for Security Operations Center (SOC) analysts.

Analyze the security alert using ONLY the evidence provided.

CURRENT ALERT:
{json.dumps(alert, indent=2)}

RELATED ALERTS:
{json.dumps(related, indent=2)}

AUTHENTICATION EVIDENCE:
{json.dumps(evidence, indent=2)}

RULE-BASED RISK ASSESSMENT:
{json.dumps(risk, indent=2)}

EVIDENCE RULES:

1. FACT
Only describe information directly present
in the evidence as FACT.

2. INFERENCE
If you make a reasonable interpretation that
is not directly proven, label it INFERENCE.

3. UNKNOWN
If evidence is insufficient, explicitly state:
"This cannot be determined from the available evidence."

4. NEVER claim that:

- credentials were compromised
- the IP belongs to an attacker
- malware spread
- lateral movement occurred
- data exfiltration occurred
- MFA was bypassed
- privilege escalation occurred

unless the evidence explicitly proves it.

5. Recommendations are suggestions for human
investigation, not confirmed facts.

Use exactly these sections:

1. What happened?
2. Why is it suspicious or normal?
3. Evidence
4. Known vs Unknown
5. Recommended Investigation

Keep the response concise and evidence-based.

The final decision must remain with a human
SOC analyst.
"""

            response = llm.invoke(prompt)

            return {
                **state,
                "analysis": response.content
            }

        except Exception as e:

            return {
                **state,
                "error":
                    f"LLM analysis failed: {str(e)}"
            }

    # ========================================================
    # NODE 6 — RECOMMENDATION
    # ========================================================

    def recommendation_node(
        state: SOCAidState
    ) -> SOCAidState:

        if state.get("error"):
            return state

        try:

            level = state[
                "risk_assessment"
            ]["risk_level"]

            recommendations = {

                "LOW":
                    "Monitor the activity and close "
                    "the alert if no additional "
                    "suspicious evidence is found.",

                "MEDIUM":
                    "Review related activity and "
                    "verify whether the activity "
                    "was expected.",

                "HIGH":
                    "Prioritize human investigation "
                    "and review the affected account, "
                    "endpoint, and related events.",

                "CRITICAL":
                    "Escalate immediately for human "
                    "investigation and review the "
                    "affected account, endpoint, "
                    "authentication logs, and "
                    "related activity."
            }

            recommendation = recommendations.get(
                level,
                "Perform human SOC analyst investigation."
            )

            return {
                **state,
                "recommendation": recommendation,
                "decision":
                    "Human SOC analyst approval required."
            }

        except Exception as e:

            return {
                **state,
                "error":
                    f"Recommendation generation failed: {str(e)}"
            }

    # ========================================================
    # CONDITIONAL ROUTING
    # ========================================================

    def route_after_parse(state):

        if state.get("error"):
            return "end"

        return "continue"

    def route_after_correlation(state):

        if state.get("error"):
            return "end"

        return "continue"

    def route_after_authentication(state):

        if state.get("error"):
            return "end"

        return "continue"

    def route_after_risk(state):

        if state.get("error"):
            return "end"

        return "continue"

    def route_after_analysis(state):

        if state.get("error"):
            return "end"

        return "continue"

    # ========================================================
    # BUILD LANGGRAPH
    # ========================================================

    workflow = StateGraph(SOCAidState)

    workflow.add_node(
        "parse_alert",
        parse_alert_node
    )

    workflow.add_node(
        "correlate_alerts",
        correlation_node
    )

    workflow.add_node(
        "authentication_evidence",
        authentication_evidence_node
    )

    workflow.add_node(
        "assess_risk",
        risk_node
    )

    workflow.add_node(
        "analyze",
        analysis_node
    )

    workflow.add_node(
        "recommend",
        recommendation_node
    )

    # --------------------------------------------------------
    # ENTRY
    # --------------------------------------------------------

    workflow.set_entry_point(
        "parse_alert"
    )

    # --------------------------------------------------------
    # PARSE → CORRELATE
    # --------------------------------------------------------

    workflow.add_conditional_edges(
        "parse_alert",
        route_after_parse,
        {
            "continue": "correlate_alerts",
            "end": END
        }
    )

    # --------------------------------------------------------
    # CORRELATE → AUTHENTICATION EVIDENCE
    # --------------------------------------------------------

    workflow.add_conditional_edges(
        "correlate_alerts",
        route_after_correlation,
        {
            "continue": "authentication_evidence",
            "end": END
        }
    )

    # --------------------------------------------------------
    # AUTHENTICATION EVIDENCE → RISK
    # --------------------------------------------------------

    workflow.add_conditional_edges(
        "authentication_evidence",
        route_after_authentication,
        {
            "continue": "assess_risk",
            "end": END
        }
    )

    # --------------------------------------------------------
    # RISK → ANALYSIS
    # --------------------------------------------------------

    workflow.add_conditional_edges(
        "assess_risk",
        route_after_risk,
        {
            "continue": "analyze",
            "end": END
        }
    )

    # --------------------------------------------------------
    # ANALYSIS → RECOMMENDATION
    # --------------------------------------------------------

    workflow.add_conditional_edges(
        "analyze",
        route_after_analysis,
        {
            "continue": "recommend",
            "end": END
        }
    )

    # --------------------------------------------------------
    # RECOMMENDATION → HUMAN DECISION
    # --------------------------------------------------------

    workflow.add_edge(
        "recommend",
        END
    )

    return workflow.compile()


# ============================================================
# REBUILD FINAL AGENT
# ============================================================

final_agent = build_soc_aid_agent(
    llm,
    alerts
)

# Keep the commonly used agent variables synchronized
soc_aid_agent = final_agent
source_agent = final_agent

print("=" * 70)
print("       ✓ FINAL SOC-Aid AGENT BUILT")
print("=" * 70)
print("✓ Alert parsing")
print("✓ Alert correlation")
print("✓ Authentication evidence")
print("✓ Risk assessment")
print("✓ Evidence-aware AI analysis")
print("✓ Recommendation engine")
print("✓ Human oversight")
print("=" * 70)

       ✓ FINAL SOC-Aid AGENT BUILT
✓ Alert parsing
✓ Alert correlation
✓ Authentication evidence
✓ Risk assessment
✓ Evidence-aware AI analysis
✓ Recommendation engine
✓ Human oversight


## 6. Agent Execution and Triage Report

In [ ]:
# SOC-Aid | Run Agent

def run_soc_aid(agent, alert: Dict[str, Any]) -> Dict[str, Any]:
    """Run SOC-Aid on a security alert."""
    return agent.invoke({"alert": alert})


selected_alert = alerts[1]
result = run_soc_aid(soc_aid_agent, selected_alert)

print("=" * 70)
print("SOC-Aid SECURITY TRIAGE")
print("=" * 70)
print(json.dumps(selected_alert, indent=2))

if result.get("error"):
    print("\n[STATUS] ERROR")
    print(result["error"])
    print("Human SOC analyst approval required.")
else:
    risk = result["risk_assessment"]
    print(f"\nRisk: {risk['risk_level']} ({risk['risk_score']}/100)")
    print("\n[AI ANALYSIS]")
    print(result["analysis"])
    print("\n[RECOMMENDATION]")
    print(result["recommendation"])
    print("\n[DECISION]")
    print(result["decision"])


SOC-Aid SECURITY TRIAGE
{
  "alert_id": "A002",
  "timestamp": "2026-08-18 01:45:32",
  "user": "admin",
  "source_ip": "185.220.101.45",
  "event_type": "successful_login",
  "message": "Successful login after multiple failed attempts",
  "severity": "high"
}

Risk: CRITICAL (90/100)

[AI ANALYSIS]
**1. What happened?**  
- The user **admin** successfully logged in from IP **185.220.101.45** at **2026‑08‑18 01:45:32** (alert A002).  
- A preceding **failed login** for the same user and IP occurred at **2026‑08‑18 01:42:10** (alert A001).  
- Authentication evidence records **1 failed login** within a **10‑minute window** and flags a **pattern detected**.

**2. Why is it suspicious or normal?**  
- *Suspicious*: A successful login immediately after a failed attempt can indicate credential guessing or a brute‑force attempt.  
- *Normal*: The single failed attempt could be a benign typo.  
- The rule‑based assessment assigns a **CRITICAL** risk score (90) based on the combination of aler

In [ ]:
# SOC-Aid | Professional Triage Report

def generate_triage_report(alert: Dict[str, Any]) -> None:
    """Run SOC-Aid and display an analyst-style triage report."""
    result = run_soc_aid(soc_aid_agent, alert)
    print("\n" + "=" * 70)
    print("SOC-Aid TRIAGE REPORT")
    print("=" * 70)
    if result.get("error"):
        print("STATUS: ERROR")
        print(result["error"])
        print("Human SOC analyst approval required.")
        return

    parsed = result["parsed_alert"]
    risk = result["risk_assessment"]
    evidence = result.get("authentication_evidence", {})
    print(f"Alert ID: {parsed['alert_id']}")
    print(f"Risk: {risk['risk_level']} ({risk['risk_score']}/100)")
    print("Authentication evidence:")
    print(json.dumps(evidence, indent=2))
    print("\nRecommendation:")
    print(result["recommendation"])
    print("\nDecision:")
    print(result["decision"])


generate_triage_report(selected_alert)



SOC-Aid TRIAGE REPORT
Alert ID: A002
Risk: CRITICAL (90/100)
Authentication evidence:
{
  "failed_login_count": 1,
  "time_window_minutes": 10,
  "pattern_detected": true,
  "first_failed_login": "2026-08-18 01:42:10",
  "last_failed_login": "2026-08-18 01:42:10"
}

Recommendation:
Escalate immediately for human investigation and review the affected account, endpoint, authentication logs, and related activity.

Decision:
Human SOC analyst approval required.


## 7. Functional Tests

In [ ]:
# SOC-Aid | Functional Testing

def run_test(test_name, alert, expected_risk=None, expect_error=False):
    print(f"\nTEST: {test_name}")
    result = run_soc_aid(soc_aid_agent, alert)

    if expect_error:
        passed = bool(result.get("error"))
    elif result.get("error"):
        print(result["error"])
        passed = False
    else:
        passed = (
            (expected_risk is None or result["risk_assessment"]["risk_level"] == expected_risk)
            and all(result.get(key) for key in [
                "parsed_alert", "risk_assessment", "analysis", "recommendation", "decision"
            ])
        )

    print("PASS" if passed else "FAIL")
    return passed


test_results = {
    "Suspicious Login": run_test("Suspicious Login", alerts[1], "CRITICAL"),
    "Normal Login": run_test("Normal Login", alerts[2], "LOW"),
    "Critical Malware": run_test("Critical Malware", alerts[3], "CRITICAL"),
    "Invalid Alert": run_test(
        "Invalid Alert",
        {"alert_id": "INVALID", "event_type": "unknown_event"},
        expect_error=True,
    ),
}

print(f"\nFunctional tests passed: {sum(test_results.values())}/{len(test_results)}")



TEST: Suspicious Login
PASS

TEST: Normal Login
PASS

TEST: Critical Malware
PASS

TEST: Invalid Alert
PASS

Functional tests passed: 4/4


## 8. Final Verification: Evidence, Risk, and Human Oversight

In [ ]:
# SOC-Aid | Final Evidence Verification

final_alert = {
    "alert_id": "SOC-FINAL-001",
    "timestamp": "2026-08-19 10:30:00",
    "user": "employee01",
    "source_ip": "192.168.1.25",
    "event_type": "successful_login",
    "message": "Successful login after repeated authentication failures",
    "severity": "high",
}
final_result = run_soc_aid(soc_aid_agent, final_alert)
evidence = final_result["authentication_evidence"]
risk = final_result["risk_assessment"]

checks = [
    evidence.get("failed_login_count") == 50,
    evidence.get("time_window_minutes") == 10,
    evidence.get("pattern_detected") is True,
    evidence.get("first_failed_login") == "2026-08-19 10:20:00",
    evidence.get("last_failed_login") == "2026-08-19 10:28:10",
    risk.get("risk_level") == "CRITICAL",
    risk.get("risk_score") == 100,
    final_result.get("decision") == "Human SOC analyst approval required.",
]

print("=" * 70)
print("SOC-Aid FINAL EVIDENCE VERIFICATION")
print("=" * 70)
print(json.dumps(evidence, indent=2))
print(f"Risk: {risk.get('risk_level')} ({risk.get('risk_score')}/100)")
print(final_result.get("decision"))
print("✓ ALL FINAL EVIDENCE CHECKS PASSED" if all(checks) else f"⚠ {sum(checks)}/{len(checks)} checks passed")


SOC-Aid FINAL EVIDENCE VERIFICATION
{
  "failed_login_count": 50,
  "time_window_minutes": 10,
  "pattern_detected": true,
  "first_failed_login": "2026-08-19 10:20:00",
  "last_failed_login": "2026-08-19 10:28:10"
}
Risk: CRITICAL (100/100)
Human SOC analyst approval required.
✓ ALL FINAL EVIDENCE CHECKS PASSED


## 9. Gradio Analyst UI

In [ ]:
!pip install -q gradio


In [ ]:
# SOC-Aid | Gradio Analyst UI

import gradio as gr

def soc_aid_ui(alert_json):
    try:
        alert = json.loads(alert_json)
        result = run_soc_aid(soc_aid_agent, alert)
        if result.get("error"):
            return ("ERROR", "", "", "", result["error"], "Human SOC analyst approval required.")

        risk = result.get("risk_assessment", {})
        evidence = result.get("authentication_evidence", {})
        risk_output = f"Risk Level: {risk.get('risk_level')}\nRisk Score: {risk.get('risk_score')}/100"
        evidence_output = (
            f"Failed Logins: {evidence.get('failed_login_count')}\n"
            f"Time Window: {evidence.get('time_window_minutes')} minutes\n"
            f"Pattern Detected: {evidence.get('pattern_detected')}\n"
            f"First Failure: {evidence.get('first_failed_login')}\n"
            f"Last Failure: {evidence.get('last_failed_login')}"
        )
        related_output = "\n".join(
            f"{item.get('alert_id')} | {item.get('event_type')} | {item.get('timestamp')} | {item.get('source_ip')}"
            for item in result.get("related_alerts", [])
        ) or "No related alerts found."
        return (
            risk_output,
            evidence_output,
            related_output,
            result.get("analysis", "No AI analysis generated."),
            result.get("recommendation", "Human SOC analyst investigation required."),
            result.get("decision", "Human SOC analyst approval required."),
        )
    except json.JSONDecodeError:
        return ("INVALID INPUT", "", "", "", "Please provide valid JSON.", "Human SOC analyst approval required.")
    except Exception as error:
        return ("ERROR", "", "", "", f"{type(error).__name__}: {error}", "Human SOC analyst approval required.")


example_json = json.dumps(final_alert, indent=2)

with gr.Blocks(title="SOC-Aid | SOC Analyst Assistant") as soc_aid_demo:
    gr.Markdown("""# 🛡️ SOC-Aid
### Explainable AI Assistant for Security Operations

⚠️ **Human analyst approval is always required before final action.**""")
    with gr.Row():
        with gr.Column():
            alert_input = gr.Code(value=example_json, language="json", label="Security Alert")
            analyze_button = gr.Button("🔍 Analyze Security Alert", variant="primary")
        with gr.Column():
            risk_output = gr.Textbox(label="🚨 Risk Assessment", lines=3)
            evidence_output = gr.Textbox(label="🔐 Authentication Evidence", lines=6)
    related_output = gr.Textbox(label="🔗 Related Security Activity", lines=8)
    analysis_output = gr.Markdown(label="🤖 Evidence-Aware AI Analysis")
    recommendation_output = gr.Textbox(label="📋 Recommended Investigation", lines=4)
    decision_output = gr.Textbox(label="👤 Human Decision", lines=2)
    analyze_button.click(
        fn=soc_aid_ui,
        inputs=alert_input,
        outputs=[risk_output, evidence_output, related_output, analysis_output, recommendation_output, decision_output],
    )

soc_aid_demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c6a45c33d3480e3536.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c6a45c33d3480e3536.gradio.live
